# Embedding Generation


1. `all-MiniLM-L6-v2` 
2. `nomic-ai/nomic-embed-text-v1` 
3. `text-embedding-3-small` 


In [ ]:
!pip install sentence-transformers openai numpy einops

In [9]:
import json
import os
import time
import numpy as np
from typing import List, Dict, Any

In [14]:
# Config
INPUT_PATH = "../chuncking/data/chunks_fixed300_overlap20.jsonl" 
OUTPUT_DIR = "./output/chunks_fixed300_overlap20"
BATCH_SIZE = 16

## Load Chunks

In [15]:
def load_chunks(path: str) -> List[Dict[str, Any]]:
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))
    print(f"Loaded {len(chunks)} chunks from {path}")
    return chunks

chunks = load_chunks(INPUT_PATH)
texts = [c["text"] for c in chunks]

Loaded 2253 chunks from ../chuncking/data/chunks_fixed300_overlap20.jsonl


In [16]:
def save_embedded_chunks(
    chunks: List[Dict],
    embeddings: np.ndarray,
    output_path: str,
):
    """Write chunks with embeddings filled in to JSONL."""
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for chunk, emb in zip(chunks, embeddings):
            record = {
                "chunk_id": chunk["chunk_id"],
                "text": chunk["text"],
                "embedding": emb.tolist(),
                "metadata": chunk["metadata"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"Saved to {output_path} ({size_mb:.2f} MB)")

## Model 1: all-MiniLM-L6-v2

Lightweight sentence-transformer (22M params, 384d). Fast and widely used as a baseline.

In [17]:
from sentence_transformers import SentenceTransformer

model_minilm = SentenceTransformer("all-MiniLM-L6-v2")

t0 = time.time()
embeddings_minilm = model_minilm.encode(
    texts, batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True
)
elapsed = time.time() - t0

print(f"Shape: {embeddings_minilm.shape}")
print(f"Time : {elapsed:.1f}s ({len(texts)/elapsed:.0f} chunks/s)")

save_embedded_chunks(
    chunks, embeddings_minilm,
    os.path.join(OUTPUT_DIR, "embedded_minilm.jsonl")
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 853.12it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 141/141 [00:33<00:00,  4.26it/s]


Shape: (2253, 384)
Time : 33.2s (68 chunks/s)
Saved to ./output/chunks_fixed300_overlap20\embedded_minilm.jsonl (21.16 MB)


## Model 2: nomic-ai/nomic-embed-text-v1

Open-source 768d model with strong MTEB scores. Requires a task prefix for documents.

In [ ]:
model_nomic = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True
)

# nomic requires prefix: "search_document: " for indexing, "search_query: " for queries
texts_prefixed = [f"search_document: {t}" for t in texts]

t0 = time.time()
embeddings_nomic = model_nomic.encode(
    texts_prefixed, batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True
)
elapsed = time.time() - t0

print(f"Shape: {embeddings_nomic.shape}")
print(f"Time : {elapsed:.1f}s ({len(texts)/elapsed:.0f} chunks/s)")

save_embedded_chunks(
    chunks, embeddings_nomic,
    os.path.join(OUTPUT_DIR, "embedded_nomic.jsonl")
)

## Summary

In [ ]:
print(f"{'Model':35s} {'Dim':>6s} {'File':>30s}")
print("-" * 75)
for name, emb, fname in [
    ("all-MiniLM-L6-v2", embeddings_minilm, "embedded_minilm.jsonl"),
    ("nomic-embed-text-v1", embeddings_nomic, "embedded_nomic.jsonl"),
    ("text-embedding-3-small", embeddings_openai, "embedded_openai_small.jsonl"),
]:
    path = os.path.join(OUTPUT_DIR, fname)
    size = os.path.getsize(path) / (1024 * 1024)
    print(f"{name:35s} {emb.shape[1]:>6d} {fname:>30s} ({size:.1f} MB)")

In [ ]:
# Verify: load first record from each file
for fname in ["embedded_minilm.jsonl", "embedded_nomic.jsonl", "embedded_openai_small.jsonl"]:
    path = os.path.join(OUTPUT_DIR, fname)
    with open(path, "r", encoding="utf-8") as f:
        rec = json.loads(f.readline())
    emb_len = len(rec["embedding"])
    print(f"[{fname}] chunk_id={rec['chunk_id']}  embedding_dim={emb_len}")

In [18]:

import faiss
import numpy as np
import json
import os

EMBEDDED_FILES = [
    "output/chunks_fixed300_overlap20/embedded_minilm.jsonl",
    "output/chunks_fixed300_overlap20/embedded_nomic.jsonl",
    "output/chunks_fixed300_overlap20/embedded_openai_small.jsonl",
]

INDEX_DIR = "output/chunks_fixed300_overlap20/"

In [19]:

def build_faiss_index(embedded_path: str, output_dir: str):
    """Load embedded JSONL, build and save FAISS index."""
    basename = os.path.splitext(os.path.basename(embedded_path))[0]

    embeddings_list = []
    with open(embedded_path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line.strip())
            if not rec.get("embedding"):
                continue
            embeddings_list.append(rec["embedding"])

    if not embeddings_list:
        print(f"[SKIP] No valid embeddings in {embedded_path}")
        return

    embeddings = np.array(embeddings_list, dtype=np.float32)
    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    os.makedirs(output_dir, exist_ok=True)
    index_path = os.path.join(output_dir, f"{basename}.faiss")
    faiss.write_index(index, index_path)

    print(f"[OK] {basename}: {index.ntotal} vectors, dim={embeddings.shape[1]} -> {index_path}")


for path in EMBEDDED_FILES:
    if not os.path.exists(path):
        print(f"[SKIP] Not found: {path}")
        continue
    build_faiss_index(path, INDEX_DIR)

[OK] embedded_minilm: 2253 vectors, dim=384 -> output/chunks_fixed300_overlap20/embedded_minilm.faiss
[OK] embedded_nomic: 2253 vectors, dim=768 -> output/chunks_fixed300_overlap20/embedded_nomic.faiss
[OK] embedded_openai_small: 2253 vectors, dim=1536 -> output/chunks_fixed300_overlap20/embedded_openai_small.faiss


In [ ]:
# Verify: load each index, query with first vector, look up text from original JSONL
for path in EMBEDDED_FILES:
    basename = os.path.splitext(os.path.basename(path))[0]
    index_path = os.path.join(INDEX_DIR, f"{basename}.faiss")
    if not os.path.exists(index_path):
        continue

    index = faiss.read_index(index_path)
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # Use first chunk as query (should match itself at rank 1)
    first = json.loads(lines[0])
    q = np.array([first["embedding"]], dtype=np.float32)
    faiss.normalize_L2(q)
    scores, indices = index.search(q, k=3)

    print(f"\n--- {basename} ({index.ntotal} vectors) ---")
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        rec = json.loads(lines[idx])
        print(f"  #{rank+1} score={score:.4f}  {rec['chunk_id']}  [{rec['metadata']['title']}]")
        print(f"       {rec['text'][:80]}...")


---
---

## Batch Embedding: All Chunking Files x All Models

Scan `../chunking/data/` for all `.jsonl` chunk files, generate embeddings + FAISS index
for each selected model. Output structure:
```
output/
  chunks_fixed300_overlap0/
    embedded_minilm.jsonl + .faiss
    embedded_nomic.jsonl  + .faiss
  chunks_fixed300_overlap20/
    ...
```

In [22]:
# ===================== BATCH CONFIG =====================

CHUNKS_DIR = "../chuncking/data"                   # directory with chunk JSONL files
OUTPUT_BASE = "./output"                          # output root
BATCH_SIZE  = 16

# Select which models to run (comment out to skip)
MODELS_TO_RUN = [
    # "minilm",
    "nomic",
    # "openai_small",                             # requires API key, skip for submission
]

# Model definitions
BATCH_MODEL_REGISTRY = {
    "minilm": {
        "model_name": "all-MiniLM-L6-v2",
        "doc_prefix": "",
        "output_tag": "embedded_minilm",
    },
    "nomic": {
        "model_name": "nomic-ai/nomic-embed-text-v1.5",
        "trust_remote_code": True,
        "doc_prefix": "search_document: ",
        "output_tag": "embedded_nomic",
    }
}

# Discover all chunk JSONL files
import glob
chunk_files = sorted(glob.glob(os.path.join(CHUNKS_DIR, "*.jsonl")))
print(f"Found {len(chunk_files)} chunk files in {CHUNKS_DIR}:")
for f in chunk_files:
    print(f"  {os.path.basename(f)}")
print(f"\nModels to run: {MODELS_TO_RUN}")
print(f"Total jobs: {len(chunk_files)} x {len(MODELS_TO_RUN)} = {len(chunk_files) * len(MODELS_TO_RUN)}")

Found 6 chunk files in ../chuncking/data:
  chunks_fixed300_overlap0.jsonl
  chunks_fixed300_overlap20.jsonl
  chunks_sentence220_overlap1.jsonl
  chunks_sentence220_overlap2.jsonl
  chunks_structure_aware_coarse.jsonl
  chunks_structure_aware_fine.jsonl

Models to run: ['nomic']
Total jobs: 6 x 1 = 6


In [23]:
def embed_and_index(
    chunks: List[Dict],
    texts: List[str],
    model: SentenceTransformer,
    model_config: dict,
    output_dir: str,
):
    """Embed texts, save JSONL + build FAISS index in one go."""
    tag = model_config["output_tag"]
    prefix = model_config.get("doc_prefix", "")
    texts_input = [prefix + t for t in texts] if prefix else texts

    # Embed
    t0 = time.time()
    embeddings = model.encode(
        texts_input, batch_size=BATCH_SIZE,
        show_progress_bar=True, normalize_embeddings=True
    )
    elapsed = time.time() - t0
    print(f"    Embedded: {embeddings.shape}, {elapsed:.1f}s ({len(texts)/elapsed:.0f} chunks/s)")

    # Save JSONL
    jsonl_path = os.path.join(output_dir, f"{tag}.jsonl")
    save_embedded_chunks(chunks, embeddings, jsonl_path)

    # Build FAISS index
    emb_np = np.array(embeddings, dtype=np.float32)
    faiss.normalize_L2(emb_np)
    index = faiss.IndexFlatIP(emb_np.shape[1])
    index.add(emb_np)
    faiss_path = os.path.join(output_dir, f"{tag}.faiss")
    faiss.write_index(index, faiss_path)
    print(f"    FAISS: {index.ntotal} vectors, dim={emb_np.shape[1]} -> {faiss_path}")

In [24]:
# Load models once
_model_cache = {}

for model_tag in MODELS_TO_RUN:
    config = BATCH_MODEL_REGISTRY[model_tag]
    if config.get("type") == "openai":
        print(f"Skipping {model_tag} (OpenAI API, handle separately)")
        continue
    kwargs = {}
    if config.get("trust_remote_code"):
        kwargs["trust_remote_code"] = True
    _model_cache[model_tag] = SentenceTransformer(config["model_name"], **kwargs)
    print(f"Loaded: {model_tag} -> {config['model_name']}")

<All keys matched successfully>


Loaded: nomic -> nomic-ai/nomic-embed-text-v1.5


In [25]:
# Run batch embedding
batch_log = []
total_jobs = len(chunk_files) * len([m for m in MODELS_TO_RUN if m in _model_cache])
job_idx = 0

for chunk_path in chunk_files:
    # Derive output dir name from chunk filename
    # e.g. chunks_fixed300_overlap20pct.jsonl -> chunks_fixed300_overlap20pct
    chunk_name = os.path.splitext(os.path.basename(chunk_path))[0]
    output_dir = os.path.join(OUTPUT_BASE, chunk_name)
    os.makedirs(output_dir, exist_ok=True)

    # Load chunks once per file
    chunks = load_chunks(chunk_path)
    texts = [c["text"] for c in chunks]

    for model_tag in MODELS_TO_RUN:
        if model_tag not in _model_cache:
            continue
        job_idx += 1
        config = BATCH_MODEL_REGISTRY[model_tag]
        print(f"\n[{job_idx}/{total_jobs}] {chunk_name} x {model_tag}")

        t0 = time.time()
        embed_and_index(chunks, texts, _model_cache[model_tag], config, output_dir)
        elapsed = time.time() - t0

        batch_log.append({
            "chunk_file": chunk_name,
            "model": model_tag,
            "n_chunks": len(chunks),
            "output_dir": output_dir,
            "time_s": round(elapsed, 2),
        })

# Save log
log_path = os.path.join(OUTPUT_BASE, "batch_embedding_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(batch_log, f, indent=2)
print(f"\nBatch complete: {len(batch_log)} jobs. Log -> {log_path}")

Loaded 1838 chunks from ../chuncking/data\chunks_fixed300_overlap0.jsonl

[1/6] chunks_fixed300_overlap0 x nomic


Batches: 100%|██████████| 115/115 [05:09<00:00,  2.69s/it]


    Embedded: (1838, 768), 309.6s (6 chunks/s)
Saved to ./output\chunks_fixed300_overlap0\embedded_nomic.jsonl (32.26 MB)
    FAISS: 1838 vectors, dim=768 -> ./output\chunks_fixed300_overlap0\embedded_nomic.faiss
Loaded 2253 chunks from ../chuncking/data\chunks_fixed300_overlap20.jsonl

[2/6] chunks_fixed300_overlap20 x nomic


Batches: 100%|██████████| 141/141 [08:37<00:00,  3.67s/it]


    Embedded: (2253, 768), 517.3s (4 chunks/s)
Saved to ./output\chunks_fixed300_overlap20\embedded_nomic.jsonl (39.51 MB)
    FAISS: 2253 vectors, dim=768 -> ./output\chunks_fixed300_overlap20\embedded_nomic.faiss
Loaded 2145 chunks from ../chuncking/data\chunks_sentence220_overlap1.jsonl

[3/6] chunks_sentence220_overlap1 x nomic


...

    Embedded: (2394, 768), 606.0s (4 chunks/s)
Saved to ./output\chunks_sentence220_overlap2\embedded_nomic.jsonl (41.92 MB)
    FAISS: 2394 vectors, dim=768 -> ./output\chunks_sentence220_overlap2\embedded_nomic.faiss
Loaded 2607 chunks from ../chuncking/data\chunks_structure_aware_coarse.jsonl

[5/6] chunks_structure_aware_coarse x nomic


Batches: 100%|██████████| 163/163 [06:20<00:00,  2.33s/it]


    Embedded: (2607, 768), 380.5s (7 chunks/s)
Saved to ./output\chunks_structure_aware_coarse\embedded_nomic.jsonl (44.69 MB)
    FAISS: 2607 vectors, dim=768 -> ./output\chunks_structure_aware_coarse\embedded_nomic.faiss
Loaded 2444 chunks from ../chuncking/data\chunks_structure_aware_fine.jsonl

[6/6] chunks_structure_aware_fine x nomic


Batches: 100%|██████████| 153/153 [06:40<00:00,  2.62s/it]


    Embedded: (2444, 768), 400.7s (6 chunks/s)
Saved to ./output\chunks_structure_aware_fine\embedded_nomic.jsonl (42.02 MB)
    FAISS: 2444 vectors, dim=768 -> ./output\chunks_structure_aware_fine\embedded_nomic.faiss

Batch complete: 6 jobs. Log -> ./output\batch_embedding_log.json


In [ ]:
# Summary table
print(f"{'Chunk File':40s} {'Model':10s} {'Chunks':>7s} {'Time':>7s}")
print('-' * 70)
for entry in batch_log:
    print(
        f"{entry['chunk_file']:40s} "
        f"{entry['model']:10s} "
        f"{entry['n_chunks']:>7d} "
        f"{entry['time_s']:>6.1f}s"
    )

# Verify output files
print(f"\nGenerated files:")
for entry in batch_log:
    tag = BATCH_MODEL_REGISTRY[entry['model']]['output_tag']
    for ext in ['.jsonl', '.faiss']:
        fpath = os.path.join(entry['output_dir'], f"{tag}{ext}")
        if os.path.exists(fpath):
            size = os.path.getsize(fpath) / (1024*1024)
            print(f"  {fpath:60s} ({size:.1f} MB)")
        else:
            print(f"  {fpath:60s} MISSING!")